# Day 2: Pandas Deep Dive
## Week 1 — Python for ML & Math Refresh

**Time:** 1.5 hours  
**Goal:** Master DataFrames, groupby, merge, pivot — the data manipulation layer between raw data and ML models.

> 💡 **Data Engineer connection:** Pandas is to Python what SQL is to databases. Your SQL instincts translate directly — SELECT → indexing, WHERE → boolean mask, GROUP BY → groupby, JOIN → merge.

---

## 1. Series & DataFrame Basics

In [ ]:
import pandas as pd
import numpy as np

# ── Series: a labelled 1D array ──
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='values')
print(f"Series:\n{s}")
print(f"\nIndex: {s.index.tolist()}")
print(f"Values: {s.values}")
print(f"Dtype: {s.dtype}")

In [ ]:
# ── DataFrame: a labelled 2D table ──
# Method 1: From dict (most common)
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'age': [28, 35, 42, 31, 26],
    'salary': [75000, 95000, 120000, 85000, 65000],
    'department': ['Engineering', 'Marketing', 'Engineering', 'Marketing', 'Engineering'],
    'years_exp': [3, 8, 15, 6, 2]
})

print(df)
print(f"\nShape: {df.shape}")       # (5, 5) — 5 rows, 5 columns
print(f"Columns: {df.columns.tolist()}")
print(f"Dtypes:\n{df.dtypes}")

In [ ]:
# ── Essential inspection methods ──
print("=== .info() ===")
df.info()

print("\n=== .describe() ===")
print(df.describe())

print(f"\n=== .head(3) ===\n{df.head(3)}")
print(f"\n=== .tail(2) ===\n{df.tail(2)}")
print(f"\n=== .sample(2) ===\n{df.sample(2, random_state=42)}")

## 2. Selecting & Filtering Data

In [ ]:
# ── Column selection ──
print(df['name'])                          # Single column → Series
print(df[['name', 'salary']])              # Multiple columns → DataFrame

# ── Row selection with .loc (label-based) and .iloc (integer-based) ──
print(f"\n.loc[0]:\n{df.loc[0]}")          # Row by label (index)
print(f"\n.iloc[0:2]:\n{df.iloc[0:2]}")    # First 2 rows by position

# ── Specific cell ──
print(f"\n.loc[2, 'name'] = {df.loc[2, 'name']}")        # Charlie
print(f".iloc[2, 0] = {df.iloc[2, 0]}")                    # Charlie

# ── Conditional filtering (WHERE clause equivalent) ──
engineers = df[df['department'] == 'Engineering']
print(f"\nEngineers:\n{engineers}")

high_earners = df[(df['salary'] > 80000) & (df['years_exp'] > 5)]
print(f"\nHigh earners with 5+ years:\n{high_earners}")

# ── .query() — SQL-like syntax ──
result = df.query("salary > 80000 and department == 'Engineering'")
print(f"\nQuery result:\n{result}")

## 3. Data Manipulation

In [ ]:
# ── Adding columns ──
df['salary_monthly'] = df['salary'] / 12
df['senior'] = df['years_exp'] >= 5
df['salary_per_year_exp'] = (df['salary'] / df['years_exp']).round(0)

print(df[['name', 'salary_monthly', 'senior', 'salary_per_year_exp']])

In [ ]:
# ── Apply: custom functions to columns ──
def categorise_salary(salary):
    if salary >= 100000:
        return 'High'
    elif salary >= 80000:
        return 'Medium'
    else:
        return 'Low'

df['salary_band'] = df['salary'].apply(categorise_salary)
print(df[['name', 'salary', 'salary_band']])

# ── Lambda shortcut ──
df['name_upper'] = df['name'].apply(lambda x: x.upper())
print(f"\n{df[['name', 'name_upper']]}")

In [ ]:
# ── Sorting ──
print("Sorted by salary (desc):")
print(df.sort_values('salary', ascending=False)[['name', 'salary']])

print("\nSorted by dept, then salary:")
print(df.sort_values(['department', 'salary'], ascending=[True, False])[['name', 'department', 'salary']])

## 4. GroupBy — Aggregation (GROUP BY equivalent)

In [ ]:
# ── Basic groupby ──
dept_stats = df.groupby('department')['salary'].agg(['mean', 'min', 'max', 'count'])
print(f"Department salary stats:\n{dept_stats}")

# ── Multiple aggregations ──
summary = df.groupby('department').agg({
    'salary': ['mean', 'sum'],
    'years_exp': 'mean',
    'name': 'count'
}).round(0)
print(f"\nFull summary:\n{summary}")

In [ ]:
# ── GroupBy transform: apply function but keep original shape ──
# (Like a window function in SQL)
df['dept_avg_salary'] = df.groupby('department')['salary'].transform('mean')
df['salary_vs_dept'] = ((df['salary'] / df['dept_avg_salary'] - 1) * 100).round(1)
print(df[['name', 'department', 'salary', 'dept_avg_salary', 'salary_vs_dept']])

## 5. Merge / Join — Combining DataFrames

In [ ]:
# ── Create two related tables ──
employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'dept_id': [101, 102, 101, 103, 102]
})

departments = pd.DataFrame({
    'dept_id': [101, 102, 103, 104],
    'dept_name': ['Engineering', 'Marketing', 'Sales', 'HR']
})

# ── Inner join (default) ──
inner = pd.merge(employees, departments, on='dept_id')
print(f"Inner join:\n{inner}")

# ── Left join ──
left = pd.merge(employees, departments, on='dept_id', how='left')
print(f"\nLeft join:\n{left}")

# ── Right join ── 
right = pd.merge(employees, departments, on='dept_id', how='right')
print(f"\nRight join:\n{right}")

# ── Outer join ──
outer = pd.merge(employees, departments, on='dept_id', how='outer')
print(f"\nOuter join:\n{outer}")

## 6. Handling Missing Data

In [ ]:
# ── Create data with missing values ──
messy = pd.DataFrame({
    'name': ['Alice', 'Bob', None, 'Diana', 'Eve'],
    'age': [28, np.nan, 42, 31, np.nan],
    'salary': [75000, 95000, np.nan, 85000, 65000]
})

print(f"Messy data:\n{messy}")
print(f"\nNull counts:\n{messy.isnull().sum()}")
print(f"\nNull percentage:\n{(messy.isnull().mean() * 100).round(1)}%")

# ── Drop nulls ──
print(f"\nDropna (any):\n{messy.dropna()}")
print(f"\nDropna (subset=['salary']):\n{messy.dropna(subset=['salary'])}")

# ── Fill nulls ──
filled = messy.copy()
filled['age'] = filled['age'].fillna(filled['age'].median())
filled['salary'] = filled['salary'].fillna(filled['salary'].mean())
filled['name'] = filled['name'].fillna('Unknown')
print(f"\nFilled:\n{filled}")

## 7. Pivot Tables & Reshaping

In [ ]:
# ── Create sales data ──
sales = pd.DataFrame({
    'month': ['Jan', 'Jan', 'Feb', 'Feb', 'Mar', 'Mar'] * 2,
    'region': ['North', 'South'] * 6,
    'product': ['A', 'A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B', 'B'],
    'revenue': [100, 150, 120, 160, 130, 170, 200, 250, 220, 260, 230, 270]
})

print(f"Sales data:\n{sales}")

# ── Pivot table ──
pivot = sales.pivot_table(
    values='revenue', 
    index='product', 
    columns='month', 
    aggfunc='sum'
)
print(f"\nPivot table:\n{pivot}")

# ── With margins (totals) ──
pivot2 = sales.pivot_table(
    values='revenue', 
    index='region', 
    columns='product', 
    aggfunc='sum', 
    margins=True
)
print(f"\nWith totals:\n{pivot2}")

## 8. String Operations & Datetime

In [ ]:
# ── String methods via .str accessor ──
names = pd.Series(['  Alice Smith  ', 'bob jones', 'CHARLIE BROWN', 'diana-prince'])

print(f"strip:      {names.str.strip().tolist()}")
print(f"lower:      {names.str.lower().tolist()}")
print(f"title:      {names.str.strip().str.title().tolist()}")
print(f"contains:   {names.str.contains('a', case=False).tolist()}")
print(f"split first:{names.str.strip().str.split().str[0].tolist()}")
print(f"len:        {names.str.strip().str.len().tolist()}")

In [ ]:
# ── Datetime ──
dates = pd.DataFrame({
    'date_str': ['2024-01-15', '2024-03-22', '2024-07-04', '2024-12-25'],
    'value': [100, 200, 300, 400]
})

dates['date'] = pd.to_datetime(dates['date_str'])
dates['year'] = dates['date'].dt.year
dates['month'] = dates['date'].dt.month
dates['day_name'] = dates['date'].dt.day_name()
dates['quarter'] = dates['date'].dt.quarter

print(dates[['date', 'year', 'month', 'day_name', 'quarter', 'value']])

## 9. Reading/Writing Data

In [ ]:
# ── CSV (most common) ──
# df.to_csv('output.csv', index=False)
# df = pd.read_csv('data.csv')

# ── Excel ──
# df.to_excel('output.xlsx', index=False)
# df = pd.read_excel('data.xlsx', sheet_name='Sheet1')

# ── JSON ──
# df.to_json('output.json', orient='records')
# df = pd.read_json('data.json')

# ── Parquet (best for large datasets — your data engineering format!) ──
# df.to_parquet('output.parquet', engine='pyarrow')
# df = pd.read_parquet('data.parquet')

# ── Quick demo with CSV string ──
from io import StringIO
csv_data = """name,age,city
Alice,28,London
Bob,35,Manchester
Charlie,42,Edinburgh"""

df_csv = pd.read_csv(StringIO(csv_data))
print(df_csv)

## 10. Practice Exercises

In [ ]:
# ── Exercise 1: Create a DataFrame of 100 random "employees"
# Columns: id, salary (40000-150000), department (randomly from 5 depts), rating (1-5)
# Find: avg salary per department, highest rated department ──

np.random.seed(42)
depts = ['Engineering', 'Marketing', 'Sales', 'HR', 'Finance']
emp = pd.DataFrame({
    'id': range(1, 101),
    'salary': np.random.randint(40000, 150000, 100),
    'department': np.random.choice(depts, 100),
    'rating': np.random.randint(1, 6, 100)
})

print("Avg salary per dept:")
print(emp.groupby('department')['salary'].mean().round(0).sort_values(ascending=False))

print(f"\nHighest rated dept: {emp.groupby('department')['rating'].mean().idxmax()}")

In [ ]:
# ── Exercise 2: Method chaining — clean and analyse in one pipeline ──
result = (emp
    .assign(salary_k=lambda x: x['salary'] / 1000)
    .query('rating >= 3')
    .groupby('department')
    .agg(avg_salary_k=('salary_k', 'mean'),
         count=('id', 'count'),
         avg_rating=('rating', 'mean'))
    .sort_values('avg_salary_k', ascending=False)
    .round(1)
)
print(result)

## Quick Reference: Pandas ↔ SQL Mapping

| SQL | Pandas |
|-----|--------|
| `SELECT col1, col2` | `df[['col1', 'col2']]` |
| `WHERE col > 5` | `df[df['col'] > 5]` |
| `GROUP BY col` | `df.groupby('col')` |
| `ORDER BY col DESC` | `df.sort_values('col', ascending=False)` |
| `JOIN ... ON` | `pd.merge(df1, df2, on='key')` |
| `COUNT(*)` | `df.groupby('col').size()` |
| `HAVING` | Filter after groupby |
| `DISTINCT` | `df['col'].unique()` |
| `CASE WHEN` | `np.where()` or `.apply()` |
| `WINDOW FUNC` | `.groupby().transform()` |

---
**✅ Day 2 Complete!** Tomorrow: Data Visualisation with Matplotlib & Seaborn.